In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

from xgboost import XGBRegressor

In [2]:
# Load data
df = pd.read_csv("df_x_hour_btc_labeled.csv")
df

,datetime,price_open,price_close,price_high,price_low,price_vol,price_vol_weight_avg,future_pct_change_5,return_1,return_3,...,close_lag_1,return_lag_1,close_lag_2,return_lag_2,close_lag_3,return_lag_3,close_lag_6,return_lag_6,close_lag_12,return_lag_12
0,2026-03-23 22:00:00,70857.00,70752.28,70904.00,70479.01,456.868569,70648.718584,0.004900,-0.002084,0.003678,...,70900.01,0.003403,70659.55,0.002363,70492.99,0.000983,70387.00,0.001961,71258.01,0.001659
1,2026-03-23 21:00:00,70895.10,70840.26,71012.00,70804.69,237.524311,70907.318664,-0.003377,0.001243,0.002557,...,70752.28,-0.002084,70900.01,0.003403,70659.55,0.002363,70543.70,0.002226,71303.34,0.000636
2,2026-03-23 20:00:00,70655.95,70901.44,70995.00,70554.91,436.801876,70771.266454,-0.011247,0.000864,0.000020,...,70840.26,0.001243,70752.28,-0.002084,70900.01,0.003403,70423.78,-0.001700,70944.41,-0.005034
3,2026-03-23 19:00:00,71061.80,70652.68,71260.00,70614.77,673.936687,70946.949797,0.011260,-0.003509,-0.001408,...,70901.44,0.000864,70840.26,0.001243,70752.28,-0.002084,70492.99,0.000983,71335.98,0.005519
4,2026-03-23 18:00:00,71067.48,71074.81,71212.00,70856.41,677.764882,71034.647691,0.005616,0.005975,0.003311,...,70652.68,-0.003509,70901.44,0.000864,70840.26,0.001243,70659.55,0.002363,70517.08,-0.011479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4050,2025-10-06 03:00:00,124100.10,123958.35,124180.00,123700.00,196.800676,123971.344644,-0.005876,0.002721,-0.000215,...,123622.02,0.000008,123621.00,-0.002936,123985.00,0.004500,124009.46,-0.000028,125060.76,-0.003778
4051,2025-10-06 02:00:00,124089.00,124168.00,124289.80,123522.30,357.934503,123913.495631,-0.011285,0.001691,0.004425,...,123958.35,0.002721,123622.02,0.000008,123621.00,-0.002936,123887.55,-0.000983,124536.10,-0.004195
4052,2025-10-06 01:00:00,123383.20,124067.32,124290.94,123231.15,419.938971,123836.235212,-0.010705,-0.000811,0.003602,...,124168.00,0.001691,123958.35,0.002721,123622.02,0.000008,123429.52,-0.003697,125086.03,0.004416
4053,2025-10-06 00:00:00,123520.80,123388.64,124432.61,123115.77,1082.891475,123608.841754,-0.006781,-0.005470,-0.004596,...,124067.32,-0.000811,124168.00,0.001691,123958.35,0.002721,123985.00,0.004500,124600.24,-0.003884


## 1) Define your target

In [3]:
target_col = "future_pct_change_5"

df = df.copy()
df = df.drop_duplicates()

# Remove rows where target is missing
df = df.dropna(subset=[target_col])

X = df.drop(columns=[target_col])
y = df[target_col]

In [4]:
X

,datetime,price_open,price_close,price_high,price_low,price_vol,price_vol_weight_avg,return_1,return_3,return_6,...,close_lag_1,return_lag_1,close_lag_2,return_lag_2,close_lag_3,return_lag_3,close_lag_6,return_lag_6,close_lag_12,return_lag_12
0,2026-03-23 22:00:00,70857.00,70752.28,70904.00,70479.01,456.868569,70648.718584,-0.002084,0.003678,0.005190,...,70900.01,0.003403,70659.55,0.002363,70492.99,0.000983,70387.00,0.001961,71258.01,0.001659
1,2026-03-23 21:00:00,70895.10,70840.26,71012.00,70804.69,237.524311,70907.318664,0.001243,0.002557,0.004204,...,70752.28,-0.002084,70900.01,0.003403,70659.55,0.002363,70543.70,0.002226,71303.34,0.000636
2,2026-03-23 20:00:00,70655.95,70901.44,70995.00,70554.91,436.801876,70771.266454,0.000864,0.000020,0.006783,...,70840.26,0.001243,70752.28,-0.002084,70900.01,0.003403,70423.78,-0.001700,70944.41,-0.005034
3,2026-03-23 19:00:00,71061.80,70652.68,71260.00,70614.77,673.936687,70946.949797,-0.003509,-0.001408,0.002265,...,70901.44,0.000864,70840.26,0.001243,70752.28,-0.002084,70492.99,0.000983,71335.98,0.005519
4,2026-03-23 18:00:00,71067.48,71074.81,71212.00,70856.41,677.764882,71034.647691,0.005975,0.003311,0.005877,...,70652.68,-0.003509,70901.44,0.000864,70840.26,0.001243,70659.55,0.002363,70517.08,-0.011479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4050,2025-10-06 03:00:00,124100.10,123958.35,124180.00,123700.00,196.800676,123971.344644,0.002721,-0.000215,-0.000412,...,123622.02,0.000008,123621.00,-0.002936,123985.00,0.004500,124009.46,-0.000028,125060.76,-0.003778
4051,2025-10-06 02:00:00,124089.00,124168.00,124289.80,123522.30,357.934503,123913.495631,0.001691,0.004425,0.002264,...,123958.35,0.002721,123622.02,0.000008,123621.00,-0.002936,123887.55,-0.000983,124536.10,-0.004195
4052,2025-10-06 01:00:00,123383.20,124067.32,124290.94,123231.15,419.938971,123836.235212,-0.000811,0.003602,0.005167,...,124168.00,0.001691,123958.35,0.002721,123622.02,0.000008,123429.52,-0.003697,125086.03,0.004416
4053,2025-10-06 00:00:00,123520.80,123388.64,124432.61,123115.77,1082.891475,123608.841754,-0.005470,-0.004596,-0.004810,...,124067.32,-0.000811,124168.00,0.001691,123958.35,0.002721,123985.00,0.004500,124600.24,-0.003884


## 2) Identify numeric and categorical columns

In [5]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['price_open', 'price_close', 'price_high', 'price_low', 'price_vol', 'price_vol_weight_avg', 'return_1', 'return_3', 'return_6', 'return_12', 'log_return', 'body', 'range', 'upper_wick', 'lower_wick', 'body_ratio', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'rsi', 'ma_5', 'ma_10', 'ma_diff', 'volatility_5', 'volatility_10', 'volume_change', 'volume_ma_5', 'volume_ratio', 'price_vs_vwap', 'vwap_ratio', 'close_lag_1', 'return_lag_1', 'close_lag_2', 'return_lag_2', 'close_lag_3', 'return_lag_3', 'close_lag_6', 'return_lag_6', 'close_lag_12', 'return_lag_12']
Categorical features: ['datetime', 'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'day_of_week_0', 'day_of_week

## 3) Build preprocessing

In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 4) Split the data

In [7]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

## 5) Build the XGBoost pipeline

In [8]:
# model = XGBRegressor(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=6,
#     min_child_weight=3,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_alpha=0.0,
#     reg_lambda=1.0,
#     random_state=42,
#     n_jobs=-1
# )

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,              # ↓ BIG reduction
    min_child_weight=8,       # ↑ more conservative splits
    subsample=0.7,            # ↑ randomness
    colsample_bytree=0.7,
    gamma=1.0,                # ↑ penalize splits
    reg_alpha=1.0,            # ↑ L1 regularization
    reg_lambda=10.0,          # ↑ L2 regularization
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

## 6) Train the model

In [9]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['price_open', 'price_close',
                                                   'price_high', 'price_low',
                                                   'price_vol',
                                                   'price_vol_weight_avg',
                                                   'return_1', 'return_3',
                                                   'return_6', 'return_12',
                                                   'log_return', 'body',
                                                   'range', 'upper_wick',
                                                   'lower_wick', 'body_ratio',
                                                   'rolling_mean_3'...
                              feature_types=None, gamma=1.0, gpu_id=None,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=3, max_leaves=None, min_child_weight=8,
                              missing=nan, monotone_constraints=None,
                              n_estimators=300, n_jobs=-1,
                              num_parallel_tree=None, predictor=None,
                              random_state=42, ...))])

## 7) Generate predictions

In [10]:
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

## 8) Output performance metrics

In [11]:
def regression_metrics(y_true, y_pred, label="Set"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Avoid divide-by-zero issues for MAPE
    nonzero_mask = y_true != 0
    if nonzero_mask.sum() > 0:
        mape = mean_absolute_percentage_error(y_true[nonzero_mask], y_pred[nonzero_mask])
    else:
        mape = np.nan
    
    print(f"--- {label} Metrics ---")
    print(f"MAE:   {mae:.4f}")
    print(f"MSE:   {mse:.4f}")
    print(f"RMSE:  {rmse:.4f}")
    print(f"R²:    {r2:.4f}")
    print(f"MAPE:  {mape:.4%}" if not np.isnan(mape) else "MAPE:  N/A")
    print()

regression_metrics(y_train, y_pred_train, label="Train")
regression_metrics(y_test, y_pred_test, label="Test")

--- Train Metrics ---
MAE:   0.0085
MSE:   0.0002
RMSE:  0.0125
R²:    -0.0014
MAPE:  163.4723%

--- Test Metrics ---
MAE:   0.0078
MSE:   0.0001
RMSE:  0.0110
R²:    -0.0001
MAPE:  146.3203%



## 9) Compare actual vs predicted

In [12]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred_test,
    "error": y_test - y_pred_test,
    "abs_error": np.abs(y_test - y_pred_test)
})

results.head(10)

,actual,predicted,error,abs_error
3244,-0.003468,0.001109,-0.004577,0.004577
3245,0.000488,0.001109,-0.000622,0.000622
3246,-0.000433,0.001109,-0.001542,0.001542
3247,0.008589,0.001109,0.007480,0.007480
3248,0.005321,0.001109,0.004212,0.004212
3249,0.006156,0.001109,0.005047,0.005047
3250,0.002795,0.001109,0.001686,0.001686
3251,0.003921,0.001109,0.002812,0.002812
3252,-0.002447,0.001109,-0.003556,0.003556
3253,0.001632,0.001109,0.000523,0.000523


In [ ]:
# worst predictions
results.sort_values("abs_error", ascending=False).head(20)

## 10) Cross-validation score

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_scores = -cv_scores
print("CV RMSE scores:", rmse_scores)
print("Mean CV RMSE:", rmse_scores.mean())
print("Std CV RMSE:", rmse_scores.std())

## 11) Feature importance

In [ ]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
fitted_model = pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

## 12) Save the model

In [ ]:
# import joblib

# joblib.dump(pipeline, "xgb_regressor_pipeline.pkl")

# loaded_model = joblib.load("xgb_regressor_pipeline.pkl")
# preds = loaded_model.predict(X_test)